# Viscosities

In [ ]:
viscosity_path = "viscosities.csv"
pv_path = "process_variables.csv"
out_path = None

In [ ]:
from itertools import cycle

import matplotlib.pyplot as plt
import pandas as pd

MARKERS = {
    "G50": "o",
    "G45": "v",
    "G40": "^",
    "G40+IPA": "s",
}

viscosity_dict = {k: v for k, v in pd.read_csv(viscosity_path).groupby("slurry")}
pv_dict = {k: v for k, v in pd.read_csv(pv_path).groupby("slurry")}

fig, ax = plt.subplots()
colors = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])

for (slurry, df), color in zip(viscosity_dict.items(), colors):
    shear_rates = pv_dict[slurry]["shear_rate"].values

    descending = df["sweep_direction"] == "descending"
    in_process_range = df["shear_rate"].between(
        shear_rates.min(), shear_rates.max(), inclusive="both"
    )

    # Draw the complete curve faintly first, then redraw the measurements
    # covered by actual process conditions at full opacity.
    ax.loglog(
        df["shear_rate"][descending].values,
        df["viscosity"][descending].values,
        marker=MARKERS[slurry],
        color=color,
        alpha=0.25,
    )
    ax.loglog(
        df["shear_rate"][descending & in_process_range].values,
        df["viscosity"][descending & in_process_range].values,
        marker=MARKERS[slurry],
        color=color,
        label=slurry,
    )

ax.set_xlabel("Shear Rate (1/s)")
ax.set_ylabel("Viscosity (Pa·s)")
ax.legend()

if out_path is not None:
    plt.savefig(out_path)
else:
    fig.show()